# Task 1 - CoreML Export & Quantization Pipeline

This notebook runs a deployment pipeline for Visual Entailment:
- loads (or optionally trains) a ViT + text encoder + fusion model
- exports the vision encoder to ONNX (opset 14)
- applies INT8 post-training quantization with onnxruntime
- converts the full-precision fusion head to CoreML with coremltools
- benchmarks PyTorch vs quantized hybrid pipeline across batch sizes 1, 4, 8

Artifacts are written to `task1_artifacts/`.


In [ ]:
# Install these first if missing in your environment:
# %pip install onnx onnxruntime coremltools


In [ ]:
import importlib
import task1_coreml_pipeline as t1

importlib.reload(t1)
Task1Config = t1.Task1Config
run_task1 = t1.run_task1

config = Task1Config(
    train_csv='master_augmented_snli_ve_train.csv',
    val_csv='cleaned_snli_ve_dev.csv',
    test_csv='cleaned_snli_ve_test.csv',
    checkpoint_path='final_sota_visual_entailment3.pth',
    output_dir='task1_artifacts',
    train_enabled=False,   # set True only if you want to re-train now
    epochs=2,
    benchmark_batch_sizes=(1, 4, 8),
    benchmark_repeats=20,
    benchmark_warmup=5,
)
config


In [ ]:
summary = run_task1(config)
summary


In [ ]:
import json
import pandas as pd
from pathlib import Path

out_dir = Path(config.output_dir)

with (out_dir / 'task1_summary.json').open() as f:
    summary = json.load(f)

print('Artifacts:')
for k, v in summary['artifacts'].items():
    print(f'  {k}: {v}')

bench = pd.read_csv(out_dir / 'benchmark_results.csv')
bench


In [ ]:
import pandas as pd
from pathlib import Path

bench = pd.read_csv(Path(config.output_dir) / 'benchmark_results.csv')

pivot_latency = bench.pivot(index='batch_size', columns='engine', values='latency_mean_ms')
pivot_throughput = bench.pivot(index='batch_size', columns='engine', values='throughput_items_per_sec')

print('Mean latency (ms):')
display(pivot_latency)

print('Throughput (items/sec):')
display(pivot_throughput)
